In [11]:
from langchain.llms import OpenAI
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores.pgvector import PGVector, DistanceStrategy
from langchain.retrievers import ContextualCompressionRetriever
from langchain.chat_models import ChatOpenAI
from langchain.retrievers.document_compressors.chain_extract import LLMChainExtractor
from langchain.chains import RetrievalQA, ConversationalRetrievalChain
from langchain.retrievers.merger_retriever import MergerRetriever
from langchain.callbacks.base import BaseCallbackHandler, AsyncCallbackHandler
from langchain.memory import ConversationBufferMemory
from langchain.schema import Document
from langchain.prompts import PromptTemplate
from IPython.display import Markdown, display, JSON
import re
import json

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

In [12]:
import dotenv
import os
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from psycopg.conninfo import make_conninfo

dotenv.load_dotenv()
connection_string = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_DATABASE')}"
conn = psycopg2.connect(connection_string)

In [20]:
with psycopg2.connect(connection_string) as conn:
    q = "select * from qualifiers"
    qual = []
    with conn.cursor() as cur:
        cur.execute(q)
        c = cur.fetchall()
        qual = [q for q in c]

In [ ]:
e = embeddings.embed_query("price expansion path")
with psycopg2.connect(connection_string) as conn:
    q = "select * from qualifiers"
    qual = []
    with conn.cursor() as cur:
        cur.execute(q)
        c = cur.fetchall()
        qual = [q for q in c]

In [ ]:
def make_retrievers(key = '350', retrieval_k = 10):
    retrievers = {}
    for collection in ['paper', 'book', 'blog', 'lecture', 'notes']:
        db = PGVector(
            embedding_function=embeddings,
            connection_string=connection_string,
            collection_name=collection + "_" + key,
        )
        retrievers[collection] = db.as_retriever(search_kwargs={"k": retrieval_k})
    base_retriever = MergerRetriever(retrievers=[i for i in retrievers.values()])
    return (retrievers, base_retriever)

r_350 = make_retrievers('350', 10)
r_750 = make_retrievers('750', 10)
r_1500 = make_retrievers('1500', 10)
r_3000 = make_retrievers('3000', 10)

In [ ]:
def get_docs(docs, _type = None):
    if _type is None:
        return docs
    else:
        return [i for i in docs if i.metadata['type'] == _type]

all_docs_350 = r_350[1].get_relevant_documents(questions[0][1])
all_docs_750 = r_750[1].get_relevant_documents(questions[0][1])
all_docs_1500 = r_1500[1].get_relevant_documents(questions[0][1])
all_docs_3000 = r_3000[1].get_relevant_documents(questions[0][1])

In [ ]:
s = "Consider a competitive market with linear supply and demand curves. Assume the demand curve slopes downward as usual. If the supply curve slopes downward as well, the market will definitely reach an equilibrium price and quantity when supply and demand intersect."
def get_docs(docs, _type = None):
    if _type is None:
        return docs
    else:
        return [i for i in docs if i.metadata['type'] == _type]

all_docs_350 = r_350[1].get_relevant_documents(s)
all_docs_750 = r_750[1].get_relevant_documents(s)
all_docs_1500 = r_1500[1].get_relevant_documents(s)
all_docs_3000 = r_3000[1].get_relevant_documents(s)

In [ ]:
def get_doc(addr):
    q = "select * from questions where addr='{}'".format(addr)
    with conn.cursor() as cur:
        cur.execute(q)
        return cur.fetchone()
doc = get_doc('2016f.1.03')
doc_refs_350 = r_350[1].get_relevant_documents(doc[1])
doc_refs_750 = r_750[1].get_relevant_documents(doc[1])
doc_refs_1500 = r_1500[1].get_relevant_documents(doc[1])
doc_refs_3000 = r_3000[1].get_relevant_documents(doc[1])

In [ ]:
all_docs_750[0]

In [ ]:
for i in get_docs(all_docs_1500):
    display(i.metadata)
    display(Markdown(re.sub("\n+", "\n", i.page_content)))
    print("------")


# Asking questions without document set

In [ ]:
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")

messages = [{"role": "system", "content": "You are a PhD student in the Economics department. You are teacher assistant for microeconomics class."}]

def query_llm(query):
    messages.append({"role": "user", "content": query})
    r = openai.ChatCompletion.create(
            model="gpt-4",
            messages=messages)
    messages.append({"role": r["choices"][0]["message"]["role"], "content": r["choices"][0]["message"]["content"]})
    return messages[-1]["content"]

In [ ]:
answer = "Let A represent 'minimum wage raises unemployment' and B represent 'AER article finds that minimum wage does not raise unemployment'. We need to find P(A) using P(A|B), P(~B|A), and P(~B|~A). P(~B|A) = 1 - P(B|A). Using Bayes' Law, we can calculate P(A) for both yourself and your friend: For you: .9 = (.25P(A))/(.25P(A)+.75(1-P(A))), which implies P(A) = 0.964. For your friend: .45=(.25P(A))/(.25P(A)+.75(1-P(A))) , which implies P(A) = 0.771. By observation, .964 is not equal to 2 * 0.771. The question's statement is false. It's important to note that the text mistakenly gives the probabilities for when the article finds the minimum wage raises unemployment, while the article actually finds the minimum wage does not raise unemployment. The statement's error implies that our confidence in the minimum wage causing unemployment increases despite seeing conflicting evidence."
messages = [{"role": "system", "content": "You are a PhD student in the Economics department. You are teacher assistant for microeconomics class. Your school follows Austrian economics."}]
display(Markdown(query_llm(f"Answer this question:  {questions[0][1]}")))
print("-----")
display(Markdown(query_llm(f"analyze this answer for the question: {answer}")))

In [ ]:
# biased as austrian
messages = [{"role": "system", "content": "You are a PhD student in the Economics department. You are assistant for microeconomics class. you are biased towards austrian economics."}]
display(Markdown(query_llm(f"Answer this question:  {questions[0][1]}")))
print("-----")
display(Markdown(query_llm(f"analyze this answer for the question, explain in details what you disagree with it: {answer}")))

In [ ]:
messages = [{"role": "system", "content": "You are a PhD student in the Economics department. You are teacher assistant for microeconomics class."}]
display(Markdown(query_llm(f"Answer this question, assume that my friend and I increase our belief in minimum wage raises unemployment after reading the article:\n\n{questions[0][1]}")))
display(Markdown(query_llm(f"analyze this answer for the question, explain in details what you disagree with it: {answer}")))

# Asking questions with Document Sets

In [ ]:
import sys
class MyCustomHandlerOne(BaseCallbackHandler):
    def on_llm_new_token(self, token, **kwargs):
        print(token, end="")
        sys.stdout.flush()

    def on_llm_end(self, outputs, **kwargs):
        print("\n\n")

llm = ChatOpenAI(temperature = 0.2, model = 'gpt-4', callbacks=[MyCustomHandlerOne()], streaming=True)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=r_350[1],
    return_source_documents=True,
    verbose=False,
)
retval = qa_chain({"query": questions[0][1]})
display(Markdown(retval["result"]))

In [ ]:
llm = ChatOpenAI(temperature = 0.2, model = 'gpt-3.5-turbo-16k', callbacks=[MyCustomHandlerOne()], streaming=True)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=r_350[1],
    return_source_documents=True,
    verbose=False,
)
retval = qa_chain({"query": questions[0][1]})
display(Markdown(retval["result"]))

In [ ]:
stuff_prompt_template = """You are student in the Economics PhD, assistant for microeonomics.

This is information from your references: {context}

Answer considering only the reference material: {question}

Lecture and expand concepts required to answer.
"""
# stuff_prompt_template = """You are student in the Economics PhD, assistant for microeonomics. Use the following pieces of context to answer the question at the end.

# {context}

# Question: {question}
# Helpful Answer:"""
PROMPT = PromptTemplate(
    template=stuff_prompt_template, input_variables=["context", "question"]
)
llm = ChatOpenAI(temperature = 0.2, model = 'gpt-3.5-turbo-16k', callbacks=[MyCustomHandlerOne()], streaming=True)
# llm = ChatOpenAI(temperature = 0.2, model = 'gpt-4', callbacks=[MyCustomHandlerOne()], streaming=True)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=r_350[1],
    return_source_documents=True,
    verbose=True,
    chain_type_kwargs={"prompt": PROMPT},
)
retval = qa_chain({"query": questions[0][1]})
display(Markdown(retval["result"]))

### The exact answer with 3.5, is this a fluke?

In [ ]:
stuff_prompt_template = """You are student in the Economics PhD, assistant for microeonomics.

This is information from your references: {context}

Answer considering only the reference material: {question}

Lecture and expand concepts required to answer.
"""
# stuff_prompt_template = """You are student in the Economics PhD, assistant for microeonomics. Use the following pieces of context to answer the question at the end.

# {context}

# Question: {question}
# Helpful Answer:"""
PROMPT = PromptTemplate(
    template=stuff_prompt_template, input_variables=["context", "question"]
)
llm = ChatOpenAI(temperature = 0.2, model = 'gpt-4', callbacks=[MyCustomHandlerOne()], streaming=True)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=r_350[1],
    return_source_documents=True,
    verbose=True,
    chain_type_kwargs={"prompt": PROMPT},
)
retval = qa_chain({"query": questions[0][1]})
display(Markdown(retval["result"]))

### GPT4 insists in getting it wrong almost every time

In [ ]:
stuff_prompt_template = """You are student in the Economics PhD, assistant for microeonomics.

Use only this information: {context}

Answer considering only the reference material: {question}

Lecture and expand concepts required to answer.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
"""
# stuff_prompt_template = """You are student in the Economics PhD, assistant for microeonomics. Use the following pieces of context to answer the question at the end.

# {context}

# Question: {question}
# Helpful Answer:"""
PROMPT = PromptTemplate(
    template=stuff_prompt_template, input_variables=["context", "question"]
)
llm = ChatOpenAI(temperature = 0.2, model = 'gpt-4', callbacks=[MyCustomHandlerOne()], streaming=True)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=r_350[1],
    return_source_documents=True,
    verbose=True,
    chain_type_kwargs={"prompt": PROMPT},
)
retval = qa_chain({"query": questions[0][1]})
display(Markdown(retval["result"]))

In [ ]:
mr_prompt_template = """You are student in the Economics PhD, assistant for microeonomics. Use the following portion of a long document to see if any of the text is relevant to answer the question.
When input text is relevant, return lecture about the relation between question and input. Otherwise reply "No comment".

Document: {context}

Question: {question}

Comment:
"""

mr_combine_prompt_template = """You are student in the Economics PhD, assistant for microeonomics.
Given the following extracted parts of a long document and a question, create a final answer.
Lecture, give examples and present and explain concepts of microeconomics contained in your final answer.

SOURCES:

QUESTION: {question}
=========
{summaries}
=========
FINAL ANSWER:"""

QUESTION_PROMPT = PromptTemplate(
    template=mr_prompt_template, input_variables=["context", "question"]
)
COMBINE_PROMPT = PromptTemplate(
    template=mr_combine_prompt_template, input_variables=["summaries", "question"]
)
llm = ChatOpenAI(temperature = 1.2, model = 'gpt-4', callbacks=[MyCustomHandlerOne()], streaming=True)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="map_reduce",
    retriever=r_1500[0]["lecture"],
    return_source_documents=True,
    verbose=False,
    chain_type_kwargs={"question_prompt": QUESTION_PROMPT, "combine_prompt": COMBINE_PROMPT},
)
retval = qa_chain({"query": questions[0][1]})
display(Markdown(retval["result"]))